# Plan d.v -- Decision Branch (Modeling Path Selection)

A documented decision memo, not a computation: takes the order-of-integration classification
produced in step iii and applies the three-way branch rule from the requirements doc to
determine the modeling path, *before* any estimation happens (step vi).

See `docs/2_plan/analysis/v_decision_branch.md` for the full spec.

**Input** (`outputs/`):
- `adf_stationarity_results.csv` (from step iii, final/resolved order-of-integration column)

**Outputs** (`outputs/`):
- `modeling_path_decision.csv` -- one row: branch taken, per-variable classifications, which
  model(s) to estimate.


In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve().parents[1]
OUTPUT_DIR = ROOT / "outputs"

ADF_RESULTS_IN = OUTPUT_DIR / "adf_stationarity_results.csv"
DECISION_OUT = OUTPUT_DIR / "modeling_path_decision.csv"


## Step 1 -- Read the final order-of-integration classification per variable

In [2]:
adf_results = pd.read_csv(ADF_RESULTS_IN)

# One classification per variable -- the "order_of_integration" column is constant across every
# order_tested/test row for a given variable (step iii resolves it once, robustness rows included).
# EXR_yoy_pct only appears if step iii's depreciation-rate fallback triggered; it did not here.
classification = adf_results.groupby("variable")["order_of_integration"].agg(lambda s: s.unique())
assert (classification.map(len) == 1).all(), "a variable has inconsistent order-of-integration values across rows"
classification = classification.map(lambda arr: arr[0])

MODEL_VARIABLES = ["ERI", "DIVP", "DIVM", "INF", "EXR", "log(FDI)"]
classification = classification.reindex(MODEL_VARIABLES)
classification


variable
ERI         I(0)
DIVP        I(1)
DIVM        I(1)
INF         I(0)
EXR         I(1)
log(FDI)    I(1)
Name: order_of_integration, dtype: str

## Step 2 -- Apply the branch rule

- **Branch A -- all six I(0):** static OLS on levels only (Ch. 3.5, Eq. 3.1) is sufficient. No
  ARDL needed.
- **Branch B -- mixed I(0)/I(1), no I(2):** estimate ARDL(p,q) bounds testing as the primary
  model. Also estimate, for comparison: (a) the literal-thesis static OLS on levels, and (b) OLS
  on first-differenced variables.
- **Branch C -- any variable I(2):** do not proceed with ARDL. Stop; report back which
  variable(s) are I(2) before any estimation (step vi) can proceed.


In [3]:
i0_vars = classification[classification == "I(0)"].index.tolist()
i1_vars = classification[classification == "I(1)"].index.tolist()
i2_vars = classification[classification == "I(2)"].index.tolist()

print("I(0):", i0_vars)
print("I(1):", i1_vars)
print("I(2):", i2_vars)

if i2_vars:
    branch = "C"
elif i1_vars:
    branch = "B"
else:
    branch = "A"

print(f"\nBranch: {branch}")


I(0): ['ERI', 'INF']
I(1): ['DIVP', 'DIVM', 'EXR', 'log(FDI)']
I(2): []

Branch: B


## Step 3 -- Record the decision

In [4]:
if branch == "A":
    models_to_estimate = "Static OLS on levels only (Eq. 3.1)."
    memo = (
        "Branch A: all six variables are I(0), so static OLS on levels is sufficient. "
        "No ARDL needed."
    )
elif branch == "B":
    models_to_estimate = (
        "ARDL(p,q) bounds testing (primary model); static OLS on levels (literal-thesis "
        "comparison); OLS on first-differenced variables (comparison)."
    )
    memo = (
        f"Branch B: mixed I(0)/I(1), no I(2) regressor. I(0): {', '.join(i0_vars)}. "
        f"I(1): {', '.join(i1_vars)}. ARDL bounds testing is estimated as the primary model, "
        "with static-OLS-on-levels and OLS-on-first-differences reported alongside for "
        "comparison in the Chapter 4 write-up. The I(1) variables above are the ones driving "
        "the spurious-regression risk in the static OLS model and the ones whose long-run "
        "coefficients are meaningful in the ARDL model."
    )
else:
    models_to_estimate = "STOPPED -- no model estimated."
    memo = (
        f"Branch C: I(2) variable(s) detected -- {', '.join(i2_vars)}. ARDL bounds testing is "
        "invalid with an I(2) regressor. Stopping before step vi. This is an explicit "
        "'don't guess' stop point: how to handle the I(2) variable (further differencing, "
        "dropping, transforming) is not decided here and must be put to the user."
    )

print(memo)


Branch B: mixed I(0)/I(1), no I(2) regressor. I(0): ERI, INF. I(1): DIVP, DIVM, EXR, log(FDI). ARDL bounds testing is estimated as the primary model, with static-OLS-on-levels and OLS-on-first-differences reported alongside for comparison in the Chapter 4 write-up. The I(1) variables above are the ones driving the spurious-regression risk in the static OLS model and the ones whose long-run coefficients are meaningful in the ARDL model.


In [5]:
decision_record = pd.DataFrame([{
    "branch": branch,
    "i0_variables": ", ".join(i0_vars),
    "i1_variables": ", ".join(i1_vars),
    "i2_variables": ", ".join(i2_vars),
    "models_to_estimate": models_to_estimate,
    "memo": memo,
}])
decision_record.to_csv(DECISION_OUT, index=False)
print(f"Written -> {DECISION_OUT}")
decision_record.T


Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/modeling_path_decision.csv


,0
branch,B
i0_variables,"ERI, INF"
i1_variables,"DIVP, DIVM, EXR, log(FDI)"
i2_variables,
models_to_estimate,"ARDL(p,q) bounds testing (primary model); stat..."
memo,"Branch B: mixed I(0)/I(1), no I(2) regressor. ..."


## Decision memo -- Branch B (mixed I(0)/I(1), no I(2))

**Classification driving this decision (from step iii, EXR resolved via the Phillips-Perron
robustness check after the literal autolag="AIC" ADF gave a provisional, lag-selection-fragile
I(2) call -- see step iii's Step 8b/8c):**

| Variable | Order of integration |
|---|---|
| ERI | I(0) |
| INF | I(0) |
| DIVP | I(1) |
| DIVM | I(1) |
| EXR | I(1) |
| log(FDI) | I(1) |

No variable is I(2), so Branch C (stop) does not trigger. Not all six are I(0), so Branch A
(static OLS alone) does not apply either. This is a clean **Branch B**: two I(0) variables and
four I(1) variables, exactly the mixed case the requirements doc anticipated given the visible
trends in EXR and log(FDI).

**Modeling path for step vi:**
1. **ARDL(p,q) bounds testing -- primary model.** Valid because no regressor is I(2); the mix of
   I(0)/I(1) regressors is precisely what ARDL bounds testing is designed to handle.
2. **Static OLS on levels (literal-thesis spec, Eq. 3.1) -- reported alongside for comparison.**
   Carries spurious-regression risk given the I(1) regressors (DIVP, DIVM, EXR, log(FDI)), which
   is exactly why it is not treated as the primary model here.
3. **OLS on first-differenced variables -- reported alongside for comparison.**

All three are presented side by side in the Chapter 4 write-up, with ARDL identified as primary
and the OLS variants as robustness/comparison context.

**Note for step vi:** DIVP, DIVM, EXR, and log(FDI) are the I(1) regressors -- these carry the
spurious-regression risk in the static-OLS comparison model and are the ones whose long-run
coefficients are the meaningful quantities to interpret in the ARDL model.

No new judgment call was made in this step beyond applying the pre-agreed branch rule from the
requirements doc.


## Addendum (2026-07-17) -- Primary model redesignated after step vi/vii evidence

**This supersedes the "ARDL is primary" call above** -- not the Branch B classification itself
(the I(0)/I(1)/I(2) split above is still correct, and ARDL was still valid to *attempt*, since
no regressor is I(2)), but the downstream question of which fitted model's coefficients are
actually trusted for H1/H2 inference. That question could not be answered at this step (it
depends on evidence step v does not have access to), so it is resolved here retroactively based
on step vi (estimation) and step vii (post-estimation diagnostics) results, both already run:

1. **The ARDL bounds test did not confirm cointegration, at either specification tried.**
   - Grid-search AIC(1,2) (step vi, `ardl_bounds_test.csv`): F=2.8230 vs. 5% bounds
     [2.328, 3.500] -- between bounds (inconclusive), and below the 1% lower bound (2.957).
   - Leanest capped ARDL(1,1) (step vi, `ardl_capped_1_1_bounds_test.csv`, run specifically to
     rule out a degrees-of-freedom power problem behind the AIC-pick's inconclusive result):
     F=2.3658 vs. 5% bounds [2.328, 3.500] -- still between bounds, still below the 1% lower
     bound (2.957).
   - Both specifications agree: cointegration is not confirmed. The long-run coefficients ARDL
     would otherwise offer for H1/H2 are not on solid footing.
2. **Step vii's Breusch-Godfrey test flags uncorrected serial correlation in the capped
   ARDL(1,1) at lag 2** (`diagnostics_model_b_ardl_capped.csv`: LM(1)=3.319, p=0.0685 --
   borderline; LM(2)=6.841, p=0.0327 -- significant at 5%). The lag-1 result (matching the
   model's own (p,q)=(1,1) structure) is borderline; the lag-2 sensitivity check is not. Combined
   with (1), this undermines confidence in ARDL(1,1)'s coefficients as the basis for inference.
3. **The first-differenced OLS, by contrast, passed its full diagnostic battery cleanly**
   (`diagnostics_model_diff_ols.csv`, step vii): Breusch-Pagan, Jarque-Bera and Ramsey RESET all
   clean at 5%; Durbin-Watson inconclusive at 5% against the exact Savin-White bounds (same
   caveat as Model A, not a rejection) but with no lagged-dependent-variable confound and no
   Breusch-Godfrey lag-sensitivity problem the way ARDL(1,1) has.

**Decision:** the **first-differenced OLS is redesignated as the primary model for H1/H2
inference.** ARDL(1,1) is retained and reported as a **secondary/exploratory** specification --
its long-run coefficients are still shown in the Chapter 4 write-up (they remain informative
about direction and magnitude), but explicitly caveated as not confirmed by the bounds test,
rather than treated as the primary evidentiary basis for H1/H2. Model A (static OLS on levels)
remains the literal-thesis baseline comparison, unchanged.

This is exactly the kind of "judgment call that changes the model materially" the requirements
doc (Additional requirements) says should be flagged rather than guessed on -- flagged here, and
made on the user's explicit direction after reviewing the step vi/vii evidence above, not
unilaterally.

`models_to_estimate` and `memo` below are rewritten to reflect this; `branch`, `i0_variables`,
`i1_variables`, `i2_variables` are unchanged, since the I(0)/I(1)/I(2) classification itself is
still correct and unaffected by this redesignation.

In [6]:
# Redesignation based on step vi (estimation) + step vii (post-estimation diagnostics) evidence,
# both already run by the time this addendum was written -- see markdown above for the full
# rationale. Overwrites the same DECISION_OUT file so later steps read the corrected designation.
assert branch == "B", "this addendum's rationale is specific to the Branch B redesignation"

models_to_estimate_v2 = (
    "OLS on first-differenced variables (primary model for H1/H2 inference, redesignated "
    "2026-07-17 -- see Addendum below); static OLS on levels (literal-thesis comparison, "
    "unchanged); ARDL(1,1) bounds testing (secondary/exploratory -- bounds test inconclusive at "
    "both the grid-searched and leanest specifications; Breusch-Godfrey LM(2) flags uncorrected "
    "serial correlation at 5%)."
)
memo_v2 = (
    f"Branch B: mixed I(0)/I(1), no I(2) regressor. I(0): {', '.join(i0_vars)}. "
    f"I(1): {', '.join(i1_vars)}. ARDL bounds testing was estimated and attempted as the primary "
    "model per the original branch rule (valid to attempt since no regressor is I(2)), but is "
    "REDESIGNATED to secondary/exploratory as of 2026-07-17: the bounds test did not confirm "
    "cointegration at either the grid-searched AIC(1,2) or leanest capped ARDL(1,1) "
    "specification (step vi), and step vii's Breusch-Godfrey LM(2) test flags uncorrected serial "
    "correlation in the capped ARDL(1,1) at 5% (p=0.0327). The first-differenced OLS is now the "
    "primary model for H1/H2 inference -- it passed its full step vii diagnostic battery cleanly "
    "(Breusch-Pagan, Jarque-Bera, Ramsey RESET all clean at 5%). Static OLS on levels remains the "
    "literal-thesis baseline comparison, unchanged. See the Addendum markdown cell above and "
    "step vii's diagnostics CSVs for the full evidence trail."
)

decision_record_v2 = pd.DataFrame([{
    "branch": branch,
    "i0_variables": ", ".join(i0_vars),
    "i1_variables": ", ".join(i1_vars),
    "i2_variables": ", ".join(i2_vars),
    "models_to_estimate": models_to_estimate_v2,
    "memo": memo_v2,
}])
decision_record_v2.to_csv(DECISION_OUT, index=False)
print(f"Re-written -> {DECISION_OUT} (primary model redesignated to first-differenced OLS)")
decision_record_v2.T

Re-written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/modeling_path_decision.csv (primary model redesignated to first-differenced OLS)


,0
branch,B
i0_variables,"ERI, INF"
i1_variables,"DIVP, DIVM, EXR, log(FDI)"
i2_variables,
models_to_estimate,OLS on first-differenced variables (primary mo...
memo,"Branch B: mixed I(0)/I(1), no I(2) regressor. ..."
